#  **DNA Sequence Analyser**

A small bioinformatics tool that reads a **FASTA** file and reports, for every sequence:

* **Length**: Numeber of Bases
* **No. Of Nucleotide Bases**: A,T,G,C,(and other ambigious bases like N)
* **GC Content**: Percentage of (G + C) bases in sequence
* **Reverse Complement**: Sequence of the opposite DNA strand (5'→3')
* **Translated Protein**: Amino acid sequence using the standard codon table

**Libraries:** `Biopython`, `Pandas`

> The sequences used below are **synthetic examples** generated for demonstration. Replace `file_path` with your own FASTA file to analyse real data.

## 1. Setup
Install Biopython if needed, then import the libraries.
- `SeqIO` reads sequence files (FASTA, GenBank, etc.)
- `gc_fraction` returns the proportion of G and C bases (0 to 1)
* `pandas` used to store the data in a table format and export the given results in the form of a csv file.

In [2]:
#!pip install biopython    # uncomment if running somewhere without these installed
import pandas as pd
from Bio import SeqIO
from Bio.SeqUtils import gc_fraction

## 2. Parse the FASTA file
`SeqIO.parse()` yields one `SeqRecord` per `>` entry. `record.seq` is a Biopython `Seq` object, so it comes with built-in methods like `reverse_complement()` and `translate()`.

The parsing is wrapped in `try/except` so a wrong filename gives a clear message, and an empty result is caught with a plain `if`.

In [12]:
file_path = "sequence.fasta"   # change this to analyse your own file

try:
    records = list(SeqIO.parse(file_path, "fasta"))
except FileNotFoundError:
    raise SystemExit(f"File not found: {file_path}")
if not records:
    raise SystemExit("No FASTA records found. Is this a valid FASTA file?")

print(f"Loaded {len(records)} sequences from {file_path}")

Loaded 3 sequences from sequence.fasta


## 4. Analysis function
For each record the function:
1. Converts the sequence to uppercase, so lowercase bases are counted correctly.
2. Counts every nucleotide with `Counter`, and tracks non-ATGC bases separately.
3. Calculates GC content. `gc_fraction` returns 0 to 1, so it is multiplied by 100 for a percentage.
4. Builds the reverse complement.
5. Trims the sequence to a multiple of 3, then translates it. Trimming avoids Biopython's partial-codon warning.

In [13]:
def analyse_record(record):
    """Return a dictionary of analysis results for one SeqRecord."""
    seq = record.seq.upper()                      # normalise case
    standard = sum(seq.count(b) for b in "ATGC")
    trimmed = seq[: len(seq) - len(seq) % 3]      # keep whole codons only

    return {
        "id": record.id,
        "description": record.description,
        "length": len(seq),
        "A": seq.count('A'), "T": seq.count('T'),
        "G": seq.count('G'), "C": seq.count('C'),
        "other_bases": len(seq) - standard,       # N or other ambiguous codes
        "gc_percent": round(gc_fraction(seq) * 100, 2),
        "reverse_complement": str(seq.reverse_complement()),
        "protein": str(trimmed.translate()),      # stop codons appear as "*"
    }


results = [analyse_record(r) for r in records]

## 5. Report the results
Report the results of the analysis of the different gene sequences.

Print the results for each sequence.

In [14]:
for r in results:
    print(f"ID: {r['id']}")
    print(f"Description: {r['description']}")
    print(f"Length: {r['length']} bp")
    print(f"Counts  A:{r['A']}  T:{r['T']}  G:{r['G']}  C:{r['C']}  other:{r['other_bases']}")
    print(f"GC content: {r['gc_percent']}%")
    print(f"Reverse complement: {r['reverse_complement']}")
    print(f"Protein: {r['protein']}")
    print("-" * 60)

ID: NM_000058.1
Description: NM_000058.1 Homo sapiens breast cancer 1, early onset (BRCA1) mRNA
Length: 5711 bp
Counts  A:1956  T:1382  G:1274  C:1099  other:0
GC content: 41.55%
Reverse complement: TCAGTAGTGGCTGTGGGGGATCTGGGGTATCAGGTAGGTGTCCAGCTCCTGGCACTGGTAGAGTGCTACACTGTCCAACACCCACTCTCGGGTCACCACAGGTGCCTCACACATCTGCCCAATTGCATGGAAGCCATTGTCCTCTGTCCAGGCATCTGGCTGCACAACCACAATTGGGTGGACACCTGTGCCAAGGGTGAATGATGAAAGCTCCTTCACCACAGAAGCACCACACAGCTGTACCATCCATTCCAGTTGATCTGTGGGCATGTTGGTGAAGGGCCCATAGCAACAGATTTCTAGCCCCCTGAAGATCTTTCTGTCCTGGGATTCTCTTGCTCGCTTTGGACCTTGGTGGTTTCTTCCATTGACCACATCTCCTCTGACTTCAAAATCATGCTCATTCAGCATTTTTCTTTCTTTAATAGACTGGGTCACCCAGAAATAGCTAACTACCCATTTTCCTCCCGCAATTCCTAGAAAATATTTCAGTGTCCGTTCACACACAAACTCAGCATCTGTTTTCATAACAACATGAGTAGTCTCTTCAGTAATTAGATTAGTTAAAGTGATGTGGTGTTTTCTGGCAAACTTGTACACGAGCATAAATTCTTCTGGGGTCAGGCCAGACACCACCATGGACATTCTTTTGTTGACCCTTTCTGTTGAAGCTGTCAATTCTGGCTTCTCCCTGCTCACACTTTCTTCCATTGCATTATACCCAGCAGTATCAGTAGTATGAGCAGCAGCTGGACTCTGGGCAGATTCTGCAACTTTCAATTGGGGAACTTTCAATGCAGA

## 6. Summary table
A compact overview of all sequences. (Requires `pandas`, which comes with most notebook setups.)

In [15]:
df = pd.DataFrame(results)[["id", "length", "A", "T", "G", "C", "other_bases", "gc_percent"]]
df

,id,length,A,T,G,C,other_bases,gc_percent
0,NM_000058.1,5711,1956,1382,1274,1099,0,41.55
1,XM_001351561.2,3128,1421,1149,298,260,0,17.84
2,V01281.1,966,370,296,178,122,0,31.06


## 7. Export results
Save everything, including the reverse complement and protein, to a CSV file.

In [16]:
pd.DataFrame(results).to_csv("analysis_results.csv")
print("Saved analysis_results.csv")

Saved analysis_results.csv


## Possible Extensions to the Code
- Six-frame translation and ORF finding (real genes rarely start at position 0)
- Adding more features like calculating melting point and the molecular weight.
- Sliding-window GC content along a single long sequence and plotting GC content graph using matplotinb
- Allow Command-Line Tools to add file to program to run automatically.
- Adding a simple menu to choose between fasta and genbank files
